# Step 02 — protein modules, unsupervised

WGCNA is told **nothing** about the patients. Modules are sets of proteins that move together, so
blocks exist by construction rather than by having been selected to separate known groups.

**Why not select proteins first.** Picking proteins by how well they separate known groups collapses
them onto one axis: on this data the 60 limma-selected proteins correlated at median 0.73 and one
component explained 72% of their variance. There were no modules left to find.

In [ ]:
suppressMessages(library(WGCNA))
options(stringsAsFactors = FALSE); enableWGCNAThreads(4); set.seed(42)
COHORT <- "A"; POWER <- 12
X <- read.csv(sprintf("R_cohort-%s_log2_combat.csv", COHORT), row.names = 1, check.names = FALSE)
m <- read.csv(sprintf("R_cohort-%s_meta.csv", COHORT), row.names = 1, check.names = FALSE)
m <- m[rownames(X), , drop = FALSE]
dim(X)

## `minModuleSize` decides whether the interferon signature exists

The Kim *et al.* 25-gene interferon panel has 14 reagents on this SomaScan menu. Whether they land
in a module at all turns entirely on the minimum module size:

| minModuleSize | modules | unassigned (grey) | ISGs in grey | largest ISG module |
|---|---|---|---|---|
| 30 | 7  | 1757 | 6 | turquoise: 3 of 14 |
| 20 | 9  | 1740 | 6 | turquoise: 3 of 14 |
| 15 | 10 | 1728 | 6 | turquoise: 3 of 14 |
| 10 | 14 | 1677 | 5 | turquoise: 3 of 14 |
| **5** | **25** | **1585** | **1** | **royalblue: 6 of 14** |

At 30 — WGCNA's usual default for expression data — the interferon proteins scattered into grey and
the signature looked absent. It was not absent. It is a **12-protein module**, and a floor of 30
cannot represent a 12-protein module. That was a parameter choice, not a property of the data.

The cost is 25 modules instead of 7. Run the scan yourself rather than taking the table on trust.

In [ ]:
# The scan. Slow -- each fit is a full blockwiseModules run.
scan_minmod <- function(sizes = c(30, 20, 15, 10, 5)) {
  do.call(rbind, lapply(sizes, function(s) {
    net <- blockwiseModules(X, power = POWER, networkType = "signed", minModuleSize = s,
                            mergeCutHeight = 0.25, numericLabels = TRUE,
                            maxBlockSize = 8000, verbose = 0)
    mm <- labels2colors(net$colors)
    data.frame(minModuleSize = s, modules = length(unique(mm)) - 1, grey = sum(mm == "grey"))
  }))
}
# scan_minmod()      # uncomment to reproduce the table above

In [ ]:
net  <- blockwiseModules(X, power = POWER, networkType = "signed", minModuleSize = 5,
                         mergeCutHeight = 0.25, numericLabels = TRUE,
                         maxBlockSize = 8000, verbose = 0)
mods <- labels2colors(net$colors)
ME   <- moduleEigengenes(X, mods)$eigengenes
sort(table(mods), decreasing = TRUE)

## Does this agree with the published analysis of the same data?

The source paper reports **21 modules** on n = 207 SLE, with the interferon module being **red, 35
proteins, hub ISG15 (SOMAmer seq.14148.2)**. We are on n = 97, so we expect smaller and more
numerous modules. The test is whether the interferon module reproduces.

In [ ]:
# Locate the module by the SOMAmer the source paper names as its interferon hub.
# NEVER by colour: WGCNA assigns colours by module rank within a single fit, so
# "royalblue" in one run and "royalblue" in another are unrelated.
i   <- grep("14148", colnames(X))[1]
ifn <- mods[i]
rb  <- colnames(X)[mods == ifn]
cat(sprintf("ISG15 seq.14148.2 (the paper's hub) is in module '%s', %d proteins\n\n",
            ifn, length(rb)))
print(rb)
k <- cor(X[, rb], ME[[paste0("ME", ifn)]])[, 1]
cat("\nour hub by module membership (kME):", rb[which.max(k)],
    sprintf("%.3f", max(k)), "\n")

**The published interferon module is recovered.** `ivory` holds 14 proteins, 10 of them canonical
interferon-stimulated: STAT1 (two SOMAmers), DDX58, IFIT3, ISG15 (both SOMAmers), GBP1, MX1, IFIH1,
CXCL10. Its hub by module membership is **ISG15 `seq.14151.4`** at kME 0.94 — ISG15 is the hub
protein the source paper names for its own interferon module, found there on n = 207 by a different
route.

**But the module's boundary moves with the sample, and that has to be said alongside.** Earlier
draws of ~90 patients from the same cohort put these proteins in modules of 12, 23 and 39 members,
and one draw dissolved them into a 775-protein module. The module count at identical parameters has
ranged from 25 to 52. The proteins co-cluster reliably; *where the boundary falls* does not.

**WGCNA colours mean nothing across fits.** They are assigned by module rank within one run. This
module has been "royalblue", "darkorange", "blue" and now "ivory" across runs of the same pipeline —
always the same biology, never the same name. Everything here looks the module up by SOMAmer, which
is why the cell above greps for `14148` rather than naming a colour.

`modulePreservation` across cohorts B and C is what turns "recovered" into "reproducible", and it
has not been run.

In [ ]:
dir.create("artifacts", showWarnings = FALSE)
saveRDS(list(cohort = COHORT, X = X, meta = m, power = POWER, mods = mods, ME = ME,
             minModuleSize = 5, mergeCutHeight = 0.25, seed = 42),
        sprintf("artifacts/wgcna_%s.rds", COHORT))
write.csv(data.frame(protein = colnames(X), module = mods),
          sprintf("artifacts/modules_%s.csv", COHORT), row.names = FALSE)

The fit is saved. **Every later step reads this artifact and none of them refits** — that is what
makes the steps separable, cacheable, and safe to containerize one per stage.